# 00 · Data exploration
[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MR-SAIRAM-7/tokenEdge/blob/main/notebooks/00_data_exploration.ipynb)

Look at the bundled datasets before modelling: price history, data quality, return distributions, indicators and label balance for both labeling methods.

In [ ]:
# --- Setup: works in Google Colab and locally (run from the repo or its notebooks/ folder)
import os, sys
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    if not os.path.exists("/content/tokenEdge"):
        !git clone -q https://github.com/MR-SAIRAM-7/tokenEdge.git /content/tokenEdge
    %cd /content/tokenEdge
    !pip install -q -e ".[train]"
elif os.path.basename(os.getcwd()) == "notebooks":
    %cd ..
import tokenedge
print("TokenEdge", tokenedge.__version__)

> If pip upgrades numpy or pandas, Colab may ask to restart. Use **Runtime → Restart session** and run the cells again (the clone and install are skipped the second time).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tokenedge.config import DEFAULT_UNIVERSE, default_splits, asset_class_of
from tokenedge.data import get_ohlcv, quality_report

rows = []
for inst in DEFAULT_UNIVERSE:
    df = get_ohlcv(inst.symbol, inst.source, inst.interval)
    q = quality_report(df, inst.interval, inst.asset_class)
    rows.append({"symbol": inst.symbol, "interval": inst.interval, "bars": len(df),
                 "start": df.index[0].date(), "end": df.index[-1].date(),
                 "missing": q["missing_bars"], "zero_volume": q["zero_volume_bars"], "outliers_5sigma": len(q["outliers"])})
pd.DataFrame(rows)

## Price history with the default research splits

In [ ]:
SYMBOL, SOURCE, INTERVAL = "SPY", "yfinance", "1d"
df = get_ohlcv(SYMBOL, SOURCE, INTERVAL)
s = default_splits(SOURCE)
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(df.index, df["close"], lw=1, color="#2a78d6")
for (a, b), label, color in [(s.train, "train", "#2a78d6"), (s.val, "validation", "#eb6834"), ((s.test[0], df.index[-1]), "test", "#1baf7a")]:
    ax.axvspan(pd.Timestamp(a, tz="UTC"), pd.Timestamp(b, tz="UTC") if isinstance(b, str) else b, alpha=0.08, color=color, label=label)
ax.set_title(f"{SYMBOL} close with train / validation / test windows"); ax.legend(); ax.set_yscale("log")
plt.show()

## Return distribution: fat tails are why the PSR adjusts for kurtosis

In [ ]:
r = np.log(df["close"]).diff().dropna()
print(f"mean {r.mean():.5f}  std {r.std():.4f}  skew {r.skew():.2f}  excess kurtosis {r.kurt():.2f}")
fig, ax = plt.subplots(figsize=(8, 3))
ax.hist(r, bins=120, color="#2a78d6", alpha=0.8)
ax.set_title("Daily log returns"); plt.show()

## Features and label balance

In [ ]:
from tokenedge.features import build_features
from tokenedge.labeling import make_labels
from tokenedge.config import default_label_config

feats = build_features(df, "v1", INTERVAL)
display(feats.dropna().describe().T[["mean", "std", "min", "max"]].round(4))
for method in ("fixed_horizon", "triple_barrier"):
    lab = make_labels(df["close"], default_label_config(method, INTERVAL))
    print(method, lab["label"].value_counts(normalize=True).round(3).to_dict(),
          lab["barrier"].value_counts(normalize=True).round(3).to_dict() if method == "triple_barrier" else "")